In [1]:
from shared_lib.local import LOCAL_ENV, LOCAL_RUN
from shared_lib.spark import (
    get_spark_session,
)
spark = get_spark_session(
    app_name="Aggtrades Ingestion Job", master=True, jars=True, local_run=LOCAL_RUN, minio=LOCAL_ENV
)

26/04/08 16:14:43 WARN Utils: Your hostname, Nguyens-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.12 instead (on interface en0)
26/04/08 16:14:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/anhtu/.ivy2/cache
The jars for the packages stored in: /Users/anhtu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e9fde926-6d33-4cb2-82d7-3d025350d074;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central


:: loading settings :: url = jar:file:/Users/anhtu/.pyenv/versions/3.11.11/envs/spark/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 97ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   0   |   0   |   0   ||   5   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-e9fde926-6d33-4cb2-82d7-3d025350d074
	confs: [default]
	0 artifacts copied, 5 alre

In [2]:
import os

from pyspark.sql import types
data_lake_bucket = os.getenv("DATA_LAKE_BUCKET", "crypto-data-lake")
date = "2025-09-29"
event_date = "2025-09-29"
base_path = f"s3a://{data_lake_bucket}/raw_zone/aggtrades/"
read_path = f"{base_path}/date={date}/"
schema = types.StructType(
    [
        types.StructField("agg_trade_id", types.LongType(), True),
        types.StructField("price", types.DoubleType(), True),
        types.StructField("quantity", types.DoubleType(), True),
        types.StructField("first_trade_id", types.LongType(), True),
        types.StructField("last_trade_id", types.LongType(), True),
        types.StructField("timestamp", types.LongType(), True),
        types.StructField("is_buyer_maker", types.BooleanType(), True),
        types.StructField("is_best_match", types.BooleanType(), True),
    ]
)

In [3]:
df = spark.read.option("header", "false").option("basePath", base_path).schema(schema).csv(read_path)

26/04/08 16:14:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
from pyspark.sql import functions as F
df = df.withColumn("ingested_at", F.to_timestamp(F.col("ingestion_ts"), "yyyyMMdd_HHmmss'Z'")) \
    .withColumn("event_time", (F.col("timestamp") / 1_000_000).cast("timestamp")) \
    .withColumn("event_date", F.col("event_time").cast("date")) \
    .withColumn("created_at", F.current_timestamp()) \
    .filter(F.col("event_date") == event_date) \
    .drop("ingestion_ts").drop("date")
    

In [5]:
df.printSchema()
df.createOrReplaceTempView("aggtrades_raw")

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- symbol: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- created_at: timestamp (nullable = false)



In [6]:
spark.sql("""
select agg_trade_id, symbol, max(ingested_at) as latest_ingested_at
from aggtrades_raw
group by agg_trade_id, symbol
""").createOrReplaceTempView("aggtrades_deduped")

In [7]:
df_deduped = spark.sql("""
select r.*
from aggtrades_raw r
join aggtrades_deduped d on r.agg_trade_id = d.agg_trade_id and r.symbol = d.symbol and r.ingested_at = d.latest_ingested_at
""")

In [8]:
write_path = f"s3a://{data_lake_bucket}/landing_zone/aggtrades/"
df_deduped.repartition("symbol").write.mode("overwrite").partitionBy("event_date", "symbol").parquet(write_path)

In [10]:
spark.read.parquet(write_path).show(10, False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-------------------+--------------------------+--------------------------+----------+-------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingested_at        |event_time                |created_at                |event_date|symbol |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-------------------+--------------------------+--------------------------+----------+-------+
|3683225449  |112163.95|0.00146 |5264576620    |5264576620   |1759104000240929|true          |true         |2026-04-07 04:14:47|2025-09-29 00:00:00.240929|2026-04-08 09:14:46.461394|2025-09-29|BTCUSDT|
|3683225450  |112163.96|4.4E-4  |5264576621    |5264576621   |1759104000298671|false         |true         |2026-04-07 04:14:47|2025-09-29 00:00:00.298671|2026-04-08 09:14:46.461394|2025-09-29

26/04/08 16:19:50 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /private/var/folders/21/8r4f38z90tz5zgftyt7rck640000gn/T/blockmgr-223d7277-134b-4563-abcd-142e7fc533e7. Falling back to Java IO way
java.io.IOException: Failed to delete: /private/var/folders/21/8r4f38z90tz5zgftyt7rck640000gn/T/blockmgr-223d7277-134b-4563-abcd-142e7fc533e7
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBl